# Vault Headline Collector — grab EVERYTHING, sort later (token-free)

Flips the scanner's logic: **no keyword gate at all.** Pulls ALL headlines from the majors for the last
N hours, dedups, tags them (metadata only — nothing is dropped), writes **one CSV**, and auto-downloads
it in Colab.

**Workflow:** Run all cells → CSV downloads to your phone → upload the CSV to Claude → say
**"ingest headlines"** → Claude triages, summarizes, files what matters into the vault threads, and
flags which links are worth opening.

**Sources:** Google News TOPIC firehoses (Business / Tech / World — this aggregates Reuters, Bloomberg,
WSJ headlines through one free feed), the direct free feeds (CNBC, MarketWatch, WSJ RSS, Fox, ABC, CNN,
Guardian), Google News vault-topic queries (extra coverage depth, still no gate), and per-ticker Yahoo.

**Honest limits:** the notebook does NOT summarize — that needs a model and would cost tokens. It
collects the title + each feed's own summary snippet. The summarizing/sorting is Claude's job on upload;
that keeps this side free. RSS = headline + snippet, not full articles; open the link for the story.
The review cell at the bottom prints every collected headline (newest first, with links) if you'd rather
eyeball on the phone before uploading.

In [ ]:
import sys, subprocess
try:
    import feedparser
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','feedparser'])
    import feedparser
import urllib.parse, time, re, csv
from datetime import datetime, timezone
print('ready')

In [ ]:
# CONFIG ------------------------------------------------------------------
HOURS_BACK   = 24    # collect headlines newer than this many hours (set 12 for overnight-only)
MAX_PER_FEED = 40    # per-feed cap

# Google News TOPIC firehoses — the broad net (aggregates the paywalled majors' headlines)
GNEWS_TOPICS = {
  'GN-Business':  'https://news.google.com/rss/headlines/section/topic/BUSINESS?hl=en-US&gl=US&ceid=US:en',
  'GN-Tech':      'https://news.google.com/rss/headlines/section/topic/TECHNOLOGY?hl=en-US&gl=US&ceid=US:en',
  'GN-World':     'https://news.google.com/rss/headlines/section/topic/WORLD?hl=en-US&gl=US&ceid=US:en',
}

# Google News searches — extra depth on vault topics (still NOT a gate; just more coverage)
GNEWS_QUERIES = [
  'Iran Hormuz oil strait tanker', 'Micron memory DRAM chip shortage', 'AI capex hyperscaler data center',
  'Federal Reserve inflation rate cut', 'power grid datacenter electricity', 'Brent crude oil price',
  'Nvidia Broadcom semiconductor', 'Apple CXMT China memory chips', 'DRAM antitrust class action lawsuit',
  'AI token costs enterprise CFO', 'AI model prices open source', 'Strait of Hormuz transit fee tankers', 'Saudi Aramco Abqaiq Houthi attack',
  'hyperscaler earnings capex guidance',
]

# Direct free feeds
DIRECT_FEEDS = {
  'ZeroHedge':'https://cms.zerohedge.com/fullrss2.xml',
  'CNBC-top':'https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=100003114',
  'CNBC-mkts':'https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=15839069',
  'MarketWatch':'http://feeds.marketwatch.com/marketwatch/topstories/',
  'CNN-money':'http://rss.cnn.com/rss/money_latest.rss',
  'Fox-biz':'https://moxie.foxbusiness.com/google-publisher/markets.xml',
  'Fox-news':'https://moxie.foxnews.com/google-publisher/latest.xml',
  'ABC-money':'https://abcnews.go.com/abcnews/moneyheadlines',
  'ABC-intl':'https://abcnews.go.com/abcnews/internationalheadlines',
  'WSJ-mkts':'https://feeds.a.dj.com/rss/RSSMarketsMain.xml',
  'WSJ-world':'https://feeds.a.dj.com/rss/RSSWorldNews.xml',
  'Guardian-biz':'https://www.theguardian.com/business/rss',
}

TICKERS = ['MU','NVDA','AVGO','TSM','GOOGL','MSFT','AAPL','META','AMZN','TSLA','NFLX','LLY','NOW','SPY','INTC','PLTR']

# THREADS = metadata TAGS only. Nothing is filtered out by these — they just pre-sort the CSV for Claude.
THREADS = {
  'WAR/OIL/HORMUZ': ['iran','hormuz','bandar abbas','houthi','red sea','bab el-mandeb','strait','tanker',
      'basra','opec','brent','crude','oil','refinery','tehran','israel','gulf','kuwait','qeshm','ahvaz',
      'transit fee','blockade','ceasefire','stand-down','kharg','jizan','yanbu','oman','aramco','abqaiq','ras tanura'],
  'MEMORY/SEMIS':   ['micron','memory','dram','hbm','nand','sk hynix','samsung','tsmc','semiconductor',
      'chip','nvidia','broadcom','avgo','sox','wafer','cxmt','changxin','ymtc','yangtze memory','lpddr',
      'socamm','price-fixing','price fixing','antitrust'],
  'AI-CAPEX/FRAGILITY': ['ai capex','hyperscaler','data center','datacenter','capex','depreciation',
      'cover ratio','buyback','openai','anthropic','ai bubble','deleveraging','vendor financing',
      'private credit','ai debt','ai bonds','off-balance'],
  'MODEL-ECONOMICS': ['token price','token cost','inference cost','model routing','open-weight',
      'open source model','open-source model','deepseek','kimi','qwen','llama','agent swarm','cursor',
      'api price','spend limit','ai bills','ai spending','model price','fine-tune','fine tune'],
  'FED/INFLATION/RATES': ['federal reserve',' fed ','powell','warsh','cpi','ppi','inflation','rate cut',
      'rate hike','fomc','treasury yield','30-year','10-year','pce','jobless'],
  'KOREA/LEVERAGE': ['kospi','korea','margin call','levered etf','carry trade','deleverage','liquidation'],
  'POWER/GRID':     ['power grid','electricity','nuclear','smr','transformer','pjm','grid','fusion','uranium'],
  'GOLD/DEBASEMENT':['gold','debasement','devalue','dollar','stablecoin','bitcoin','remonetiz','crypto'],
  'EARNINGS':       ['earnings','guidance','revenue','profit','beats','misses','forecast'],
}
print('config loaded:', len(GNEWS_TOPICS), 'firehoses,', len(GNEWS_QUERIES), 'queries,',
      len(DIRECT_FEEDS), 'direct feeds,', len(TICKERS), 'tickers')

In [ ]:
# FETCH EVERYTHING — no gate. Tags are metadata only. -----------------------
def gnews_url(q): return 'https://news.google.com/rss/search?q='+urllib.parse.quote(q)+'&hl=en-US&gl=US&ceid=US:en'
def yahoo_url(t): return 'https://feeds.finance.yahoo.com/rss/2.0/headline?s='+t+'&region=US&lang=en-US'

def entry_time(e):
    for k in ('published_parsed','updated_parsed'):
        if getattr(e, k, None): return datetime.fromtimestamp(time.mktime(getattr(e,k)), tz=timezone.utc)
    return None

def tag_threads(text):
    t = ' ' + text.lower() + ' '
    return [name for name,kws in THREADS.items() if any(k in t for k in kws)]

def pull(url, source):
    out=[]
    try:
        f = feedparser.parse(url)
        for e in f.entries[:MAX_PER_FEED]:
            title = getattr(e,'title','').strip()
            summ  = re.sub('<[^<]+?>','',getattr(e,'summary','')).strip()[:300]
            out.append(dict(title=title, summary=summ, source=source,
                            link=getattr(e,'link',''), dt=entry_time(e),
                            tags=tag_threads(title+' '+summ)))
    except Exception as ex:
        print('  skip', source, ex)
    return out

rows=[]
print('Google News firehoses...');  [rows.extend(pull(u,s)) for s,u in GNEWS_TOPICS.items()]
print('Google News topics...');     [rows.extend(pull(gnews_url(q),'GN-q')) for q in GNEWS_QUERIES]
print('Direct feeds...');           [rows.extend(pull(u,s)) for s,u in DIRECT_FEEDS.items()]
print('Per-ticker (Yahoo)...');     [rows.extend(pull(yahoo_url(t),'YF:'+t)) for t in TICKERS]
print('Pulled', len(rows), 'rows (pre-dedup)')

In [ ]:
# DEDUP + RECENCY WINDOW ----------------------------------------------------
seen=set(); uniq=[]
for r in sorted(rows, key=lambda r:(r['dt'] or datetime(1970,1,1,tzinfo=timezone.utc)), reverse=True):
    key = re.sub('[^a-z0-9]','', r['title'].lower())[:60]
    if key in seen or not key: continue
    seen.add(key); uniq.append(r)

now = datetime.now(timezone.utc)
def fresh(r): return (r['dt'] is None) or ((now-r['dt']).total_seconds() <= HOURS_BACK*3600)
uniq = [r for r in uniq if fresh(r)]

def age(r):
    if not r['dt']: return '?'
    m=(now-r['dt']).total_seconds()/60
    return f'{int(m)}m' if m<90 else f'{int(m/60)}h'

print(len(uniq), 'unique headlines in the last', HOURS_BACK, 'hours')

In [ ]:
# WRITE CSV + AUTO-DOWNLOAD (Colab) ----------------------------------------
fn = 'vault_headlines_' + now.strftime('%Y-%m-%d_%H%M') + 'utc.csv'
with open(fn,'w',newline='',encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['utc_time','age','source','tags','title','summary','link'])
    for r in uniq:
        w.writerow([r['dt'].strftime('%Y-%m-%d %H:%M') if r['dt'] else '',
                    age(r), r['source'], '|'.join(r['tags']), r['title'], r['summary'], r['link']])
print('wrote', fn, '—', len(uniq), 'rows')

try:
    from google.colab import files
    files.download(fn)
    print('downloading... upload this file to Claude and say: ingest headlines')
except Exception:
    print('not in Colab — file saved at ./'+fn)

In [ ]:
# REVIEW CELL — every collected headline, newest first, with links ----------
from collections import Counter
c = Counter(t for r in uniq for t in r['tags'])
print('='*70)
ts = now.astimezone().strftime('%Y-%m-%d %H:%M %Z')
print(f'  ALL HEADLINES  ({ts}, last {HOURS_BACK}h, {len(uniq)} unique)')
print('  thread tags:', dict(c.most_common()))
print('='*70)
for r in uniq:
    tag = ('['+r['tags'][0].split('/')[0]+']') if r['tags'] else ''
    print(f"[{age(r):>4}] {r['title'][:100]}  — {r['source']} {tag}")
    if r['link']: print('       '+r['link'][:110])

### Notes
- **This collector drops nothing.** `THREADS` tags are a pre-sort column in the CSV, not a filter —
  a headline with no tag still lands in the CSV and the review cell.
- Retune by editing CONFIG: `HOURS_BACK` (12 = overnight only), feeds, tickers, tags.
- The old **scanner** notebook still exists for the filtered quick-read + the live-price LEVEL WATCH cell.
- X/Twitter: still no free path. Bloomberg/Reuters arrive via the Google News firehoses as headlines;
  their links may be paywalled — that's fine for triage.
- Workflow reminder: CSV → upload to Claude → "ingest headlines" → Claude summarizes, files to threads,
  and lists the 5-10 links worth opening.